# Bookstore Pipeline Demo

Full end-to-end example:
1. Generate an OWL ontology from a PostgreSQL bookstore database
2. Match it against the hand-crafted Slovenian bookstore ontology (`ontologija knjigarna.owx`)

The two ontologies use **different languages** (English vs. Slovenian), so string-based
techniques will mostly fail — which motivates the multilingual matching approach in the thesis.

## Setup
Before running, create the database:
```bash
createdb bookstore
psql -d bookstore -f resources/db/setup_bookstore.sql
```
Install extra dependency if needed:
```bash
pip install psycopg2-binary
```

In [ ]:
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "bookstore"
DB_USER     = "postgres"
DB_PASSWORD = "secret"       # change as needed

SLOVENIAN_ONTOLOGY = "resources/onto.rdf"   # hand-crafted Slovenian ontology
GENERATED_ONTOLOGY = "resources/db/bookstore_from_db.rdf"

## Step 1 — Generate ontology from the database

In [ ]:
from ontology_matching.src.db_to_ontology import DBToOntology

gen = DBToOntology(
    host=DB_HOST, port=DB_PORT,
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
)
g = gen.generate(GENERATED_ONTOLOGY)

### Inspect the generated ontology

In [ ]:
from rdflib.namespace import OWL, RDFS
import rdflib

print("=== Classes ===")
for s, _, _ in g.triples((None, rdflib.RDF.type, OWL.Class)):
    label = g.value(s, RDFS.label)
    parent = g.value(s, RDFS.subClassOf)
    parent_str = f"  (subClassOf {str(parent).split('#')[-1]})" if parent else ""
    print(f"  {label}{parent_str}")

print("\n=== Datatype Properties ===")
for s, _, _ in g.triples((None, rdflib.RDF.type, OWL.DatatypeProperty)):
    label  = g.value(s, RDFS.label)
    domain = g.value(s, RDFS.domain)
    range_ = g.value(s, RDFS.range)
    print(f"  {label}  (domain: {str(domain).split('#')[-1]}, range: {str(range_).split('#')[-1]})")

print("\n=== Object Properties ===")
for s, _, _ in g.triples((None, rdflib.RDF.type, OWL.ObjectProperty)):
    label  = g.value(s, RDFS.label)
    domain = g.value(s, RDFS.domain)
    range_ = g.value(s, RDFS.range)
    print(f"  {label}  (domain: {str(domain).split('#')[-1]}, range: {str(range_).split('#')[-1]})")

## Step 2 — Match against the Slovenian ontology

We run all four techniques and compare how many correspondences each finds.
Because the labels are in different languages, we expect low recall here.
This is the baseline that motivates synonym/translation-based matching.

In [ ]:
from ontology_matching.src.ontology_matching_process import OntologyMatcher

matcher = OntologyMatcher(GENERATED_ONTOLOGY, SLOVENIAN_ONTOLOGY)

techniques = ["levenshtein", "ngram", "cosine", "path"]
results = {}

for technique in techniques:
    print(f"\n{'='*50}")
    print(f"Technique: {technique}")
    print('='*50)
    matches = matcher.match_ontologies(technique=technique, threshold=0.5)
    results[technique] = matches
    print(f"Total matches found: {len(matches)}")

## Step 3 — Save alignments

In [ ]:
for technique, matches in results.items():
    out = f"resources/db/bookstore_{technique}_alignment.rdf"
    matcher.create_alignment_ontology(matches, out)
    print(f"Saved: {out}")

## Step 4 — Summary table

In [ ]:
import pandas as pd

rows = []
for technique, matches in results.items():
    for left_uri, (right_uri, score) in matches.items():
        rows.append({
            "technique":  technique,
            "left (EN)": left_uri.split("#")[-1],
            "right (SL)": right_uri.split("#")[-1],
            "score":      round(score, 3),
        })

df = pd.DataFrame(rows)
df.sort_values(["technique", "score"], ascending=[True, False])